#  AudioScene Analyzer
Speech transcription + emotion recognition + background sound detection → JSON

Model stack: WhisperX · PANNs · HuBERT · pyannote

In [ ]:
!pip install torch torchvision torchaudio librosa panns-inference transformers soundfile whisperx -q
!apt-get install -y ffmpeg -q

In [ ]:
import torch, librosa, json, os
import numpy as np
from pathlib import Path
from panns_inference import AudioTagging, SoundEventDetection, labels
from transformers import AutoModelForAudioClassification, AutoFeatureExtractor
import whisperx

# ── Config ──────────────────────────────────────────────────────────────────
AUDIO_PATH = "/content/drive/MyDrive/test.mp3"  # <-- change to your path
HF_TOKEN   = os.environ.get("HF_TOKEN", "")

DEVICE       = "cuda" if torch.cuda.is_available() else "cpu"
COMPUTE_TYPE = "float16" if DEVICE == "cuda" else "int8"  # float16 only on GPU

print(f"Device: {DEVICE} | Compute type: {COMPUTE_TYPE}")

## 1. Load audio

In [ ]:
# Load once at 32kHz for PANNs, resample for WhisperX
audio_32k, _ = librosa.load(AUDIO_PATH, sr=32000, mono=True)
audio_16k, _ = librosa.load(AUDIO_PATH, sr=16000, mono=True)

print(f"Duration: {len(audio_16k)/16000:.1f}s")

## 2. PANNs — Audio Tagging (clip-level)

In [ ]:
at = AudioTagging(checkpoint_path=None, device=DEVICE)
clipwise_output, embedding = at.inference(audio_32k[np.newaxis, :])

print("Top-10 audio tags:")
for i in np.argsort(clipwise_output[0])[::-1][:10]:
    print(f"  {labels[i]}: {clipwise_output[0][i]:.3f}")

## 3. PANNs — Sound Event Detection (frame-level)

In [ ]:
sed = SoundEventDetection(checkpoint_path=None, device=DEVICE)
framewise_output = sed.inference(audio_32k[np.newaxis, :])

FRAMES_PER_SEC    = 100
SEGMENT_LEN_SEC   = 1.0
frames_per_seg    = int(SEGMENT_LEN_SEC * FRAMES_PER_SEC)
total_frames      = framewise_output.shape[1]
SCORE_THRESHOLD   = 0.3

bg_results = []
for start in range(0, total_frames, frames_per_seg):
    end          = min(start + frames_per_seg, total_frames)
    mean_scores  = framewise_output[0, start:end, :].mean(axis=0)
    detected     = [
        (labels[i], float(mean_scores[i]))
        for i in range(len(mean_scores))
        if mean_scores[i] > SCORE_THRESHOLD
    ]
    if detected:
        bg_results.append({
            "start":  round(start / FRAMES_PER_SEC, 2),
            "end":    round(end   / FRAMES_PER_SEC, 2),
            "events": detected
        })

print(f"Background segments detected: {len(bg_results)}")

## 4. WhisperX — Transcription + Alignment + Diarization

In [ ]:
# Transcribe
whisper_model  = whisperx.load_model("large-v2", DEVICE, compute_type=COMPUTE_TYPE)
audio_whisper  = whisperx.load_audio(AUDIO_PATH)
result         = whisper_model.transcribe(audio_whisper, batch_size=16)

print(f"Detected language: {result['language']}")

# Align
model_a, metadata = whisperx.load_align_model(
    language_code=result["language"], device=DEVICE
)
result = whisperx.align(
    result["segments"], model_a, metadata,
    audio_whisper, DEVICE, return_char_alignments=False
)

# Diarize (speaker labels)
if not HF_TOKEN:
    print("⚠️  HF_TOKEN not set — skipping diarization. Add it in Colab Secrets.")
else:
    diarize_model   = whisperx.diarize.DiarizationPipeline(
        use_auth_token=HF_TOKEN, device=DEVICE
    )
    diarize_segments = diarize_model(audio_whisper)
    result = whisperx.assign_word_speakers(diarize_segments, result)

## 5. HuBERT — Emotion Recognition per segment

In [ ]:
emotion_model     = AutoModelForAudioClassification.from_pretrained(
    "superb/hubert-large-superb-er"
).to(DEVICE)
emotion_extractor = AutoFeatureExtractor.from_pretrained(
    "superb/hubert-large-superb-er"
)
emotion_model.eval()

for seg in result["segments"]:
    start = seg["start"]
    end   = seg["end"]
    clip  = audio_16k[int(start * 16000):int(end * 16000)]

    if len(clip) < 1000:
        seg["emotion"] = "unknown"
        continue

    inputs = emotion_extractor(
        clip, sampling_rate=16000,
        return_tensors="pt", padding=True
    ).to(DEVICE)

    with torch.no_grad():
        logits  = emotion_model(**inputs).logits
        pred_id = torch.argmax(logits).item()
        seg["emotion"] = emotion_model.config.id2label[pred_id]

print("Emotion tagging done.")

## 6. Merge → scene.json

In [ ]:
def match_bg_event(start, end, bg_events):
    """Return the top background sound label overlapping [start, end]."""
    for evt in bg_events:
        if evt["start"] <= start <= evt["end"] or evt["start"] <= end <= evt["end"]:
            return sorted(evt["events"], key=lambda x: x[1], reverse=True)[0][0]
    return "none"

scenes = [
    {
        "start":            round(seg["start"], 3),
        "end":              round(seg["end"],   3),
        "text":             seg.get("text", "").strip(),
        "speaker":          seg.get("speaker", "unknown"),
        "emotion":          seg.get("emotion", "neutral"),
        "background_scene": match_bg_event(seg["start"], seg["end"], bg_results),
        "words":            seg.get("words", [])
    }
    for seg in result["segments"]
]

Path("outputs").mkdir(exist_ok=True)
out_path = "outputs/scene.json"
with open(out_path, "w", encoding="utf-8") as f:
    json.dump(scenes, f, indent=2, ensure_ascii=False)

print(f"✅ Done! {len(scenes)} segments saved to {out_path}")
print(json.dumps(scenes[0], indent=2, ensure_ascii=False))  # preview first segment